# RavenStack SaaS Analytics
## 01 - SQL Data Quality

This notebook performs data quality validation on the RavenStack SaaS dataset before business analysis.

### Objectives
- Validate record counts
- Check primary key uniqueness
- Identify missing values
- Check duplicate records
- Validate foreign-key relationships
- Check date and business-rule consistency

## 1. Row Count Validation

Verify that all source tables were loaded into the Lakehouse with the expected number of records.

In [3]:
SELECT 'accounts' AS table_name, COUNT(*) AS row_count
FROM accounts

UNION ALL

SELECT 'subscriptions', COUNT(*)
FROM subscriptions

UNION ALL

SELECT 'feature_usage', COUNT(*)
FROM feature_usage

UNION ALL

SELECT 'support_tickets', COUNT(*)
FROM support_tickets

UNION ALL

SELECT 'churn_events', COUNT(*)
FROM churn_events;

StatementMeta(, 5b6d11f8-6525-44c5-a940-8a8f8d4d8d82, 4, Finished, Available, Finished, False)

<Spark SQL result set with 5 rows and 2 fields>

### Result

All five tables contain the expected number of records. No record loss was identified during the initial ingestion check.

## 2. Primary Key Validation

Check whether the expected primary key in each table uniquely identifies each record.

A primary key should not contain duplicate values because duplicate keys can create incorrect joins and unreliable analytical results.

In [4]:
SELECT
    'accounts' AS table_name,
    COUNT(*) AS total_rows,
    COUNT(DISTINCT account_id) AS unique_keys
FROM accounts

UNION ALL

SELECT
    'subscriptions',
    COUNT(*),
    COUNT(DISTINCT subscription_id)
FROM subscriptions

UNION ALL

SELECT
    'feature_usage',
    COUNT(*),
    COUNT(DISTINCT usage_id)
FROM feature_usage

UNION ALL

SELECT
    'support_tickets',
    COUNT(*),
    COUNT(DISTINCT ticket_id)
FROM support_tickets

UNION ALL

SELECT
    'churn_events',
    COUNT(*),
    COUNT(DISTINCT churn_event_id)
FROM churn_events;

StatementMeta(, 5b6d11f8-6525-44c5-a940-8a8f8d4d8d82, 5, Finished, Available, Finished, False)

<Spark SQL result set with 5 rows and 3 fields>

### Result

The primary key validation confirms that `account_id`, `subscription_id`, `ticket_id`, and `churn_event_id` are unique in their respective tables.

However, `feature_usage` contains 25,000 records but only 24,979 unique `usage_id` values, indicating duplicate primary key values that require further investigation before any data-cleaning decision is made.

## 3. Duplicate Primary Key Investigation

The primary key validation identified duplicate `usage_id` values in the `feature_usage` table.

Before removing or modifying any records, we will inspect the duplicated IDs to determine whether they represent identical records or distinct usage events sharing the same identifier.

In [5]:
-- Identify duplicated usage IDs

SELECT
    usage_id,
    COUNT(*) AS occurrence_count
FROM feature_usage
GROUP BY usage_id
HAVING COUNT(*) > 1
ORDER BY occurrence_count DESC, usage_id;

StatementMeta(, 5b6d11f8-6525-44c5-a940-8a8f8d4d8d82, 6, Finished, Available, Finished, False)

<Spark SQL result set with 21 rows and 2 fields>

### Result

The `feature_usage` table contains 21 duplicated `usage_id` values. Each duplicated ID occurs exactly twice.

This confirms a primary key uniqueness issue in the raw data. The duplicate records will be investigated further before deciding how they should be handled during data cleaning.

## 4. Inspect Duplicate Usage Records

The duplicate-key check identified 21 `usage_id` values that occur twice.

This step compares the complete records associated with those IDs to determine whether the duplicates are identical records or distinct usage events with the same identifier.

In [6]:
-- Inspect the complete records for duplicated usage IDs

SELECT *
FROM feature_usage
WHERE usage_id IN (
    SELECT usage_id
    FROM feature_usage
    GROUP BY usage_id
    HAVING COUNT(*) > 1
)
ORDER BY usage_id, usage_date;

StatementMeta(, 5b6d11f8-6525-44c5-a940-8a8f8d4d8d82, 7, Finished, Available, Finished, False)

<Spark SQL result set with 42 rows and 8 fields>

### Result

The 21 duplicated `usage_id` values correspond to different usage events rather than identical records.

The duplicated IDs are associated with different subscriptions, dates, features, and usage metrics. Therefore, these records should not be removed as duplicate rows.

The issue will be addressed during the data-cleaning stage by generating a reliable unique event key while preserving all valid usage records.

## 5. Full-Row Duplicate Validation

Primary key duplication does not necessarily mean that complete records are duplicated.

This step checks whether any two records contain identical values across all columns. Completely duplicated records may represent accidental duplicate ingestion and require separate treatment.

In [7]:
-- Check for completely duplicated records

SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT
        CONCAT_WS(
            '||',
            CAST(usage_id AS STRING),
            CAST(subscription_id AS STRING),
            CAST(usage_date AS STRING),
            CAST(feature_name AS STRING),
            CAST(usage_count AS STRING),
            CAST(usage_duration_secs AS STRING),
            CAST(error_count AS STRING),
            CAST(is_beta_feature AS STRING)
        )
    ) AS unique_rows
FROM feature_usage;

StatementMeta(, 5b6d11f8-6525-44c5-a940-8a8f8d4d8d82, 8, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 2 fields>

### Result

The `feature_usage` table contains 25,000 total rows and 25,000 unique rows.

No completely duplicated records were identified. Therefore, the dataset does not contain exact full-row duplicates in the `feature_usage` table.

## 6. Missing Value Validation

Missing values can affect calculations, joins, and business analysis.

This step checks the main columns in the `feature_usage` table for NULL values to determine whether any fields require cleaning or special handling.

In [8]:
SELECT
    COUNT(*) AS total_rows,
    SUM(CASE WHEN usage_id IS NULL THEN 1 ELSE 0 END) AS missing_usage_id,
    SUM(CASE WHEN subscription_id IS NULL THEN 1 ELSE 0 END) AS missing_subscription_id,
    SUM(CASE WHEN usage_date IS NULL THEN 1 ELSE 0 END) AS missing_usage_date,
    SUM(CASE WHEN feature_name IS NULL THEN 1 ELSE 0 END) AS missing_feature_name,
    SUM(CASE WHEN usage_count IS NULL THEN 1 ELSE 0 END) AS missing_usage_count,
    SUM(CASE WHEN usage_duration_secs IS NULL THEN 1 ELSE 0 END) AS missing_duration,
    SUM(CASE WHEN error_count IS NULL THEN 1 ELSE 0 END) AS missing_error_count,
    SUM(CASE WHEN is_beta_feature IS NULL THEN 1 ELSE 0 END) AS missing_beta_flag
FROM feature_usage;

StatementMeta(, 5b6d11f8-6525-44c5-a940-8a8f8d4d8d82, 9, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 9 fields>

### Result

The `feature_usage` table contains 25,000 rows, and all checked columns have 0 missing values.

No NULL values were identified in the key fields, including usage ID, subscription ID, usage date, feature name, usage metrics, error count, and beta-feature flag.

Therefore, no missing-value treatment is required for this table.

## 7. Date & Value Validation

Validate date relationships and numerical business rules to identify logically inconsistent records.

Checks include:
- Subscription end date should not be earlier than start date
- MRR and ARR should not be negative
- Seat count should not be negative
- Trial subscriptions should have consistent revenue values

In [9]:
-- Check subscription date and numerical business rules

SELECT
    COUNT(*) AS total_rows,
    SUM(CASE
        WHEN end_date IS NOT NULL AND end_date < start_date
        THEN 1 ELSE 0
    END) AS invalid_date_range,
    SUM(CASE
        WHEN mrr_amount < 0
        THEN 1 ELSE 0
    END) AS negative_mrr,
    SUM(CASE
        WHEN arr_amount < 0
        THEN 1 ELSE 0
    END) AS negative_arr,
    SUM(CASE
        WHEN seats < 0
        THEN 1 ELSE 0
    END) AS negative_seats
FROM subscriptions;

StatementMeta(, 5b6d11f8-6525-44c5-a940-8a8f8d4d8d82, 10, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 5 fields>

### Result

All 5,000 subscription records passed the date and numerical validation checks.

- Invalid date ranges: **0**
- Negative MRR values: **0**
- Negative ARR values: **0**
- Negative seat counts: **0**

No logical inconsistencies were identified in the subscription dates or key numerical business fields.

## 8. Foreign-Key / Referential Integrity Validation

Validate relationships between the main SaaS tables to ensure that foreign-key values correspond to valid records in their parent tables.

Checks include:
- Every `account_id` in `subscriptions` exists in `accounts`
- Every `subscription_id` in `feature_usage` exists in `subscriptions`
- Every `account_id` in `support_tickets` exists in `accounts`
- Every `account_id` in `churn_events` exists in `accounts`

In [10]:
-- Check foreign-key relationships

SELECT
    'subscriptions → accounts' AS relationship,
    COUNT(*) AS invalid_records
FROM subscriptions s
LEFT JOIN accounts a
    ON s.account_id = a.account_id
WHERE a.account_id IS NULL

UNION ALL

SELECT
    'feature_usage → subscriptions',
    COUNT(*)
FROM feature_usage f
LEFT JOIN subscriptions s
    ON f.subscription_id = s.subscription_id
WHERE s.subscription_id IS NULL

UNION ALL

SELECT
    'support_tickets → accounts',
    COUNT(*)
FROM support_tickets t
LEFT JOIN accounts a
    ON t.account_id = a.account_id
WHERE a.account_id IS NULL

UNION ALL

SELECT
    'churn_events → accounts',
    COUNT(*)
FROM churn_events c
LEFT JOIN accounts a
    ON c.account_id = a.account_id
WHERE a.account_id IS NULL;

StatementMeta(, 5b6d11f8-6525-44c5-a940-8a8f8d4d8d82, 11, Finished, Available, Finished, False)

<Spark SQL result set with 4 rows and 2 fields>

### Result

All four foreign-key relationship checks returned 0 invalid records:

- `subscriptions → accounts`: 0
- `feature_usage → subscriptions`: 0
- `support_tickets → accounts`: 0
- `churn_events → accounts`: 0

This confirms that the key relationships between the SaaS tables are referentially consistent, with no orphan records detected.

## 9. Duplicate Record Validation

Check whether records contain duplicate primary or business keys.

The validation focuses on:
- `account_id` in `accounts`
- `subscription_id` in `subscriptions`
- `usage_id` in `feature_usage`
- `ticket_id` in `support_tickets`
- `churn_event_id` in `churn_events`

In [11]:
-- Check duplicate key records

SELECT
    'accounts' AS table_name,
    COUNT(*) - COUNT(DISTINCT account_id) AS duplicate_keys
FROM accounts

UNION ALL

SELECT
    'subscriptions',
    COUNT(*) - COUNT(DISTINCT subscription_id)
FROM subscriptions

UNION ALL

SELECT
    'feature_usage',
    COUNT(*) - COUNT(DISTINCT usage_id)
FROM feature_usage

UNION ALL

SELECT
    'support_tickets',
    COUNT(*) - COUNT(DISTINCT ticket_id)
FROM support_tickets

UNION ALL

SELECT
    'churn_events',
    COUNT(*) - COUNT(DISTINCT churn_event_id)
FROM churn_events;

StatementMeta(, 5b6d11f8-6525-44c5-a940-8a8f8d4d8d82, 12, Finished, Available, Finished, False)

<Spark SQL result set with 5 rows and 2 fields>

### Result

The duplicate-key validation identified no duplicate keys in `accounts`, `subscriptions`, `support_tickets`, or `churn_events`.

However, `feature_usage` contains **21 duplicate `usage_id` values**.

This issue will be investigated further before the data is considered fully validated. The duplicate records may require removal or further review depending on whether they represent true duplicate events or valid repeated usage records.